<a href="https://colab.research.google.com/github/Diego-1099/Colabfiles/blob/main/Tarea_3_Sistema_basado_en_reglas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



---



# **Tarea 3. Sistema basado en reglas**

**Parte 1. Modelación conceptual (KR)**

**1. Dominio del discurso**


*   **¿Qué objetos existen?:** **Productos** (artículos del catálogo), **Clientes** (empresas B2B) y **Promociones**.
*   **¿Qué tipo de cosas son?:** Son entidades comerciales que interactúan a través de transacciones y consultas en el chatbot de WhatsApp.

**2. Predicados en Lógica de Primer Orden**

*   **Predicados unarios (propiedades):**

    *   **EsProductoPremium(x):** Indica si un producto pertenece a la categoría de alta gama.
    *   **TieneStock(x):** Indica si el producto **x** tiene existencias disponibles en el inventario.
    *   **EsClienteMoroso(x):** Indica si el cliente **x** tiene facturas pendientes de pago.


*   **Predicados binarios (relaciones)**

    *   **SolicitaCotizacion(x, y):** Relaciona al cliente **x** con el producto **y** que le interesa.
    *   **PrecioEspecial(x, y):** Define que el cliente **x** tiene un precio preferencial para el producto **y**.









---



**Parte 2. Base de conocimientos en LPO**

**3. Hechos (Mínimo 10)**
Estos son los datos específicos de tu inventario y clientes actuales:

1.   **EsProductoPremium(Jumex_Lata)**
1.   **TieneStock(Jumex_Lata)**
1.   **TieneStock(Agua_Ciel)**
1.   **TieneStock(Coca_Cola)**
1.   **EsClienteMoroso(Tienda_Abarrotes)**
1.   **SolicitaCotizacion(Grupo_Mexicano, Jumex_Lata)**
1.   **SolicitaCotizacion(Tienda_Abarrotes, Agua_Ciel)**
1.   **SolicitaCotizacion(Grupo_Mexicano, Coca_Cola)**
1.   **PrecioEspecial(Grupo_Mexicano, Jumex_Lata)**
1.   **EsProductoPremium(Coca_Cola)**

**4. Reglas en Lógica de Primer Orden (mínimo 5)**
Aquí definimos la Inteligencia del sistema. He incluido conjunciones (el simbolo **^** que significa "y") y reglas que dependen de otras para cumplir con los requisitos:

1.   **Regla 1 (Disponibilidad):** Si un cliente solicita un producto y hay
stock, la cotización es posible.

SolicitaCotizacion(x, y) ^ TieneStock(y) -> CotizacionDisponible(x, y)

1.   **Regla 2 (Descuento - Depende de R1):** Si la cotización es posible y el cliente tiene precio especial, aplicamos el descuento.

CotizacionDisponible(x, y) ^ PrecioEspecial(x, y) -> AplicarDescuento(x, y)

1.   **Regla 3 (Crédito):** Si un cliente es moroso, se debe bloquear cualquier intento de venta.

EsClienteMoroso(x) -> BloquearVenta(x)

1.   **Regla 4 (Notificación - Depende de R3):** Si la venta está bloqueada y el cliente solicita algo, el bot debe notificar al departamento de crédito.

BloquearVenta(x) ^ SolicitaCotizacion(x, y) -> NotificarCredito(x)

1.   **Regla 5 (Prioridad):** Si un producto es premium y tiene stock, el bot debe sugerirlo como venta prioritaria.

EsProductoPremium(y) ^ TieneStock(y) -> SugerirVentaPrioritaria(y)




---



**Parte 3. Normalización lógica**

**5. Forma canónica (Modus Ponens)**

Todas las reglas diseñadas en la base de conocimientos están expresadas en forma canónica de Horn (cláusulas definidas), cuya estructura general es $A_1 ∧ A_2 ∧ ... ∧ A_n → B$

En nuestro sistema, los antecedentes (la parte izquierda de la implicación) son conjunciones de predicados y el consecuente (la parte derecha) es un único predicado lógico. Por ejemplo, nuestra regla 1 se estructura como:

SolicitaCotizacion(x, y) ∧ TieneStock(y) → CotizacionDisponible(x, y)

**Razonamiento exclusivo con Modus Ponens:**

El Modus Ponens establece que si tenemos una regla P → Q y sabemos que P es verdadero, entonces podemos deducir Q. Nuestro motor de inferencia trabajará exclusivamente bajo este principio de la siguiente manera:



1.   Analizará los antecedentes de cada regla.
2.   Buscará en nuestra lista de hechos si esos antecedentes ya existen y son verdaderos para variables específicas (por ejemplo, si **x** es Grupo_Mexicano y **y** es Jumex_Lata).
1.   Al comprobar que todos los antecedentes se cumplen, el sistema disparará la regla y aplicará Modus Ponens para deducir el consecuente (ej. CotizacionDisponible(Grupo_Mexicano, Jumex_Lata)).
2.   Este nuevo hecho se agrega a la base de conocimientos, permitiendo que otras reglas (como la regla 2) se disparen en la siguiente iteración (encadenamiento hacia adelante).





---



In [1]:
# Parte 4. Implementación en Python

# 6. Representación de hechos

hechos = {
    "EsProductoPremium": {"Jumex_Lata", "Coca_Cola"},
    "TieneStock": {"Jumex_Lata", "Agua_Ciel", "Coca_Cola"},
    "EsClienteMoroso": {"Tienda_Abarrotes"},
    "SolicitaCotizacion": {("Grupo_Mexicano", "Jumex_Lata"), ("Tienda_Abarrotes", "Agua_Ciel"), ("Grupo_Mexicano", "Coca_Cola")},
    "PrecioEspecial": {("Grupo_Mexicano", "Jumex_Lata")},

    # Predicados a inferir (inicializados vacíos)
    "CotizacionDisponible": set(),
    "AplicarDescuento": set(),
    "BloquearVenta": set(),
    "NotificarCredito": set(),
    "SugerirVentaPrioritaria": set()
}

# 7. Representación de reglas

reglas = [
    # R1: SolicitaCotizacion(x, y) ^ TieneStock(y) -> CotizacionDisponible(x, y)
    {"antecedentes": [("SolicitaCotizacion", "x", "y"), ("TieneStock", "y")], "consecuente": ("CotizacionDisponible", "x", "y")},

    # R2: CotizacionDisponible(x, y) ^ PrecioEspecial(x, y) -> AplicarDescuento(x, y)
    {"antecedentes": [("CotizacionDisponible", "x", "y"), ("PrecioEspecial", "x", "y")], "consecuente": ("AplicarDescuento", "x", "y")},

    # R3: EsClienteMoroso(x) -> BloquearVenta(x)
    {"antecedentes": [("EsClienteMoroso", "x")], "consecuente": ("BloquearVenta", "x")},

    # R4: BloquearVenta(x) ^ SolicitaCotizacion(x, y) -> NotificarCredito(x)
    {"antecedentes": [("BloquearVenta", "x"), ("SolicitaCotizacion", "x", "y")], "consecuente": ("NotificarCredito", "x")},

    # R5: EsProductoPremium(y) ^ TieneStock(y) -> SugerirVentaPrioritaria(y)
    {"antecedentes": [("EsProductoPremium", "y"), ("TieneStock", "y")], "consecuente": ("SugerirVentaPrioritaria", "y")}
]

# 8. Motor de inferencia (obligatorio)

def unificar_y_evaluar(antecedentes, hechos_actuales, asignaciones_actuales):
    if not antecedentes:
        return [asignaciones_actuales]

    primer_ant = antecedentes[0]
    predicado = primer_ant[0]
    variables = primer_ant[1:]

    resultados = []

    for hecho in hechos_actuales[predicado]:
        hecho_tupla = (hecho,) if isinstance(hecho, str) else hecho
        asignaciones_nuevas = asignaciones_actuales.copy()
        coincide = True

        for var, val in zip(variables, hecho_tupla):
            if var in asignaciones_nuevas:
                if asignaciones_nuevas[var] != val:
                    coincide = False
                    break
            else:
                asignaciones_nuevas[var] = val

        if coincide:
            resultados.extend(unificar_y_evaluar(antecedentes[1:], hechos_actuales, asignaciones_nuevas))

    return resultados

def motor_inferencia_forward_chaining(hechos, reglas):
    iteracion = 1
    hubo_cambios = True

    print("Iniciando Motor de Inferencia...\n")

    while hubo_cambios:
        print(f"--- Iteración {iteracion} ---")
        hubo_cambios = False

        for i, regla in enumerate(reglas):
            antecedentes = regla["antecedentes"]
            consecuente = regla["consecuente"]
            pred_cons = consecuente[0]
            vars_cons = consecuente[1:]

            asignaciones_validas = unificar_y_evaluar(antecedentes, hechos, {})

            for asig in asignaciones_validas:
                nuevo_hecho = tuple(asig[var] for var in vars_cons)
                if len(nuevo_hecho) == 1:
                    nuevo_hecho = nuevo_hecho[0]

                if nuevo_hecho not in hechos[pred_cons]:
                    hechos[pred_cons].add(nuevo_hecho)
                    hubo_cambios = True
                    print(f"Regla {i+1} disparada! Nuevo hecho inferido: {pred_cons}({nuevo_hecho})")

        if not hubo_cambios:
            print("No se infirieron nuevos hechos. Punto fijo alcanzado.")
        iteracion += 1

    return hechos

# Ejecutar el motor
hechos_finales = motor_inferencia_forward_chaining(hechos, reglas)


Iniciando Motor de Inferencia...

--- Iteración 1 ---
Regla 1 disparada! Nuevo hecho inferido: CotizacionDisponible(('Grupo_Mexicano', 'Jumex_Lata'))
Regla 1 disparada! Nuevo hecho inferido: CotizacionDisponible(('Grupo_Mexicano', 'Coca_Cola'))
Regla 1 disparada! Nuevo hecho inferido: CotizacionDisponible(('Tienda_Abarrotes', 'Agua_Ciel'))
Regla 2 disparada! Nuevo hecho inferido: AplicarDescuento(('Grupo_Mexicano', 'Jumex_Lata'))
Regla 3 disparada! Nuevo hecho inferido: BloquearVenta(Tienda_Abarrotes)
Regla 4 disparada! Nuevo hecho inferido: NotificarCredito(Tienda_Abarrotes)
Regla 5 disparada! Nuevo hecho inferido: SugerirVentaPrioritaria(Coca_Cola)
Regla 5 disparada! Nuevo hecho inferido: SugerirVentaPrioritaria(Jumex_Lata)
--- Iteración 2 ---
No se infirieron nuevos hechos. Punto fijo alcanzado.




---



**Parte 5. Resultados y análisis**

**9. Reflexión**

1.  ¿Dónde está el conocimiento en tu sistema?
    -   El conocimiento reside explícitamente en la base de hechos y en la estructura lógica de las reglas de Lógica de Primer Orden (LPO). A diferencia de los pesos sinápticos en una red neuronal, aquí el conocimiento es simbólico, determinista y completamente legible para un humano.

2.  ¿Qué pasaría si agregas más reglas?
    -   El árbol de inferencias crecería en profundidad y anchura. El sistema podría deducir estados del mundo mucho más complejos, requiriendo posiblemente más iteraciones (pasadas del motor) para alcanzar el punto fijo, lo que aumentaría el costo computacional de la unificación de variables.

3.  ¿Qué no puede hacer este sistema?
    -   No puede manejar incertidumbre, ambigüedad ni información incompleta. Si un cliente escribe con errores ortográficos o pide "algo parecido al jugo", el sistema falla porque requiere coincidencias exactas de símbolos.

4.  ¿En qué se diferencia de un modelo de Machine Learning?
    -   Un modelo de Machine Learning es inductivo y estadístico, encuentra patrones y probabilidades en datos históricos. Este sistema basado en reglas es deductivo y formal, aplica lógica estricta sobre premisas declaradas, garantizando que si las premisas son verdaderas, la conclusión será absoluta, no probabilística.

5.  Si un hecho no está explícitamente en la base de conocimientos, pero es "obvio" para un humano. ¿Por qué el sistema no puede inferirlo? ¿Qué dice esto sobre el conocimiento implícito y el conocimiento tácito?
    -   El sistema opera bajo la "Asunción de Mundo Cerrado" (lo que no sabe que es verdadero, es falso). No tiene sentido común. Esto demuestra que el conocimiento tácito humano (el contexto obvio) es extremadamente difícil de formalizar en símbolos lógicos, siendo el principal cuello de botella de la IA simbólica clásica.

6.  ¿Qué asunciones sobre el mundo hiciste al definir tu dominio (D)? ¿Qué tipo de cosas decidiste no representar y por qué?
    -   Asumí un mundo determinista y simplificado donde las intenciones de compra son claras y directas. Decidí no representar elementos transitorios como la frustración del cliente, la logística de envío o pagos parciales, ya que la LPO es ineficiente para modelar estados intermedios continuos.

7.  ¿El motor de inferencia "razona" o solo aplica reglas? Justifica tu respuesta con base en cómo funciona tu código.
    -   Mecánicamente, solo aplica reglas. Su "razonamiento" es una simple unificación de patrones sintácticos (Pattern matching) para ejecutar el Modus Ponens iterativamente. El código evidencia que el motor no entiende el significado comercial de un "Descuento", solo sabe que el simbolo "x" coincide y, por ende, dispara la tupla consecuente.

8.  ¿Cómo se manifiesta la racionalidad (o irracionalidad) en tu sistema? ¿Está en las reglas, en los hechos, o en el proceso de inferencia?
    -   La racionalidad se manifiesta en el proceso de inferencia. Las reglas proveen el conocimiento, pero la racionalidad radica en el algoritmo de encadenamiento hacia adelante, el cual garantiza consistencia lógica y evita conclusiones que contradigan las premisas.

9.  ¿Qué tipo de errores puede cometer tu sistema aún cuando las reglas sean correctas?
    -   "Fragilidad del conocimiento". Por ejemplo, si el ERP reporta por error de sistema EsClienteMoroso(Grupo_Mexicano), la regla deductiva aplicará perfectamente y le bloqueará la venta, cometiendo un error comercial grave debido a que la lógica formal es ciega a la veracidad en el mundo real de sus premisas de entrada.

10. ¿Cómo imaginas un sistema híbrido (neurosimbólico) que combine tu motor de reglas con un modelo aprendido?
    -   El componente "neuro" (un modelo como GPT) actuaría como interfaz en WhatsApp, procesando el lenguaje natural ambiguo del cliente y extrayendo las intenciones. Luego, pasaría estas entidadees al componente "simbólico" (este motor de reglas), el cual validaría de forma determinista y segura el inventario, los precios y las políticas de crédito, evitando alucinaciones antes de responderle al cliente.

11. ¿Qué te obligo a pensar de manera diferente este proyecto respecto a programar "normal"?
    -   Obliga a cambiar de un paradigma imperativo ("haz esto, luego esto") a uno declarativo ("esto es verdad" y "estas son las condiciones lógicas"). Te fuerza a desmenuzar un proceso de ventas, demostrando que gran parte de lo que consideramos "inteligencia comercial" requiere una formalización extremadamente rigurosa.






---

